# Лабораторная работа 2, Самсонов Савелий Артёмович М8О-406Б-21

### Выбор задач и датасетов

Киноиндустрия сталкивается с серьезной проблемой при прогнозировании успеха фильмов на конкурентном рынке. Понимание ключевых факторов, влияющих на это и на получение дохода от фильма, имеет решающее значение для продюсеров, студий и инвесторов при принятии стратегических решений. Поэтому для определения переменных, влияющих на успех фильма, таких как бюджет, маркетинговые расходы, продолжительность фильма и рейтинги главных актеров, режиссеров и критиков, необходимо прогностическое моделирование. Для обучения моделей, которые могут предсказать успешность фильмов в прокате, взяты датасеты Movie_classification.csv и Movie_regression.xls, опубликованные на kaggle.

### Выбор метрик для классификации

1. Accuracy, измеряет долю правильно классифицированных экземпляров от общего числа примеров. Простая и понятная метрика, но может давать ошибочное представление для несбалансированных классов (когда классы имеют существенно отличающееся количество экземпляров).
2. Precision, измеряет долю правильных положительных предсказаний среди всех предсказанных положительных примеров.
3. Recall, измеряет долю правильно предсказанных положительных примеров среди всех реальных положительных примеров.
Precision, Recall важны в случае несбалансированных классов и при необходимости минимизировать ложные срабатывания.
4. F1-мера, является гармоническим средним между точностью и полнотой и используется для сбалансирования этих двух метрик. Комбинирует точность и полноту, идеально подходит для несбалансированных задач.

### Выбор метрик для регрессии

1. R², отношение между суммой квадратов отклонений предсказанных значений от среднего значения и суммой квадратов отклонений истинных значений от среднего. Показывает, какая доля вариации в целевой переменной объясняется моделью. Хороший показатель R² близкий к 1 означает, что модель хорошо объясняет данные, однако для некоторых типов задач (например, с незначительными отклонениями) значение R² может быть не таким информативным.
2. MAE, измеряет среднее абсолютное отклонение между предсказанными и истинными значениями. Полезна, когда важно понять, насколько в среднем модель ошибается по величине предсказанных значений. MAE не так чувствительна к выбросам, как другие метрики.
3. MSE, измеряет средний квадрат разницы между предсказанными и истинными значениями. MSE часто используется, когда важно акцентировать внимание на больших ошибках. Более чувствительна к выбросам, чем MAE, и может быть полезна, когда крупные ошибки особенно нежелательны.

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

In [2]:
import os
dataset_path = '.\\input'

for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

.\input\Movie_classification.csv
.\input\Movie_regression.xls


## 2. Создание бейзлайна и оценка качества

### Обучение модели из sklearn (для классификации) и оценка качества по выбранным метрикам

Загрузим датасет

In [3]:
df = pd.read_csv(dataset_path + "\\Movie_classification.csv")
df

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,3D_available,Time_taken,Twitter_hastags,Genre,Avg_age_actors,Num_multiplex,Collection,Start_Tech_Oscar
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,YES,109.60,223.840,Thriller,23,494,48000,1
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,NO,146.64,243.456,Drama,42,462,43200,0
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,NO,147.88,2022.400,Comedy,38,458,69400,1
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,YES,185.36,225.344,Drama,45,472,66800,1
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,NO,176.48,225.792,Drama,55,395,72400,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,492480,NO,186.96,243.584,Action,27,561,44800,0
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,482875,YES,132.24,263.296,Action,20,600,41200,0
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,532239,NO,109.56,243.824,Comedy,31,576,47800,0
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,496077,YES,158.80,303.520,Comedy,47,607,44000,0


Удалим некоторые параметры

In [4]:
if "Genre" in df:
  del df["Genre"]
if "3D_available" in df:
  del df["3D_available"]
if "Time_taken" in df:
  del df["Time_taken"]

Просмотрим информацию о значениях полей и убедимся, что все они допустимы

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Marketing expense    506 non-null    float64
 1   Production expense   506 non-null    float64
 2   Multiplex coverage   506 non-null    float64
 3   Budget               506 non-null    float64
 4   Movie_length         506 non-null    float64
 5   Lead_ Actor_Rating   506 non-null    float64
 6   Lead_Actress_rating  506 non-null    float64
 7   Director_rating      506 non-null    float64
 8   Producer_rating      506 non-null    float64
 9   Critic_rating        506 non-null    float64
 10  Trailer_views        506 non-null    int64  
 11  Twitter_hastags      506 non-null    float64
 12  Avg_age_actors       506 non-null    int64  
 13  Num_multiplex        506 non-null    int64  
 14  Collection           506 non-null    int64  
 15  Start_Tech_Oscar     506 non-null    int

Создадим выборки для обучения и тестирования

In [7]:
X1 = df.drop('Start_Tech_Oscar', axis=1)
y1 = df['Start_Tech_Oscar']

X1_train,X1_test,y1_train,y1_test = train_test_split(X1.values, y1.values, random_state = 0)

Обучение модели для классификации

In [8]:
from sklearn.linear_model import LogisticRegression, LinearRegression

sk_log_reg = LogisticRegression(max_iter=500, random_state=42)
sk_log_reg.fit(X1_train, y1_train)
sk_log_reg_pred_res = sk_log_reg.predict(X1_test)
sk_log_reg_accuracy = accuracy_score(y1_test, sk_log_reg_pred_res)
sk_log_reg_precision = precision_score(y1_test, sk_log_reg_pred_res)
sk_log_reg_recall = recall_score(y1_test, sk_log_reg_pred_res)
sk_log_reg_f1 = f1_score(y1_test, sk_log_reg_pred_res)

print(f'sk logistic regressor accuracy: {sk_log_reg_accuracy:}')
print(f'sk logistic regressor precision: {sk_log_reg_precision:}')
print(f'sk logistic regressor recall: {sk_log_reg_recall:}')
print(f'sk logistic regressor f1: {sk_log_reg_f1:}')

sk logistic regressor accuracy: 0.6535433070866141
sk logistic regressor precision: 0.7424242424242424
sk logistic regressor recall: 0.6447368421052632
sk logistic regressor f1: 0.6901408450704226


### Обучение модели из sklearn (для регрессии) и оценка качества по выбранным метрикам

Загрузим датасет

In [9]:
df2 = pd.read_csv(dataset_path + "\\Movie_regression.xls")
df2

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,3D_available,Time_taken,Twitter_hastags,Genre,Avg_age_actors,Num_multiplex,Collection
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,YES,109.60,223.840,Thriller,23,494,48000
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,NO,146.64,243.456,Drama,42,462,43200
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,NO,147.88,2022.400,Comedy,38,458,69400
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,YES,185.36,225.344,Drama,45,472,66800
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,NO,176.48,225.792,Drama,55,395,72400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,492480,NO,186.96,243.584,Action,27,561,44800
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,482875,YES,132.24,263.296,Action,20,600,41200
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,532239,NO,109.56,243.824,Comedy,31,576,47800
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,496077,YES,158.80,303.520,Comedy,47,607,44000


Просмотрим информацию о значениях полей для проверки, что все они допустимы

In [10]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Marketing expense    506 non-null    float64
 1   Production expense   506 non-null    float64
 2   Multiplex coverage   506 non-null    float64
 3   Budget               506 non-null    float64
 4   Movie_length         506 non-null    float64
 5   Lead_ Actor_Rating   506 non-null    float64
 6   Lead_Actress_rating  506 non-null    float64
 7   Director_rating      506 non-null    float64
 8   Producer_rating      506 non-null    float64
 9   Critic_rating        506 non-null    float64
 10  Trailer_views        506 non-null    int64  
 11  3D_available         506 non-null    object 
 12  Time_taken           494 non-null    float64
 13  Twitter_hastags      506 non-null    float64
 14  Genre                506 non-null    object 
 15  Avg_age_actors       506 non-null    int

Удалим некоторые параметры

In [11]:
del df2['3D_available']
del df2['Genre']
del df2['Time_taken']
df2.head()

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,Twitter_hastags,Avg_age_actors,Num_multiplex,Collection
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,223.840,23,494,48000
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,243.456,42,462,43200
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,2022.400,38,458,69400
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,225.344,45,472,66800
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,225.792,55,395,72400


Создадим выборки для обучения и тестирования

In [12]:
X2 = df2.drop('Collection', axis=1)
y2 = df2['Collection']

X2_train,X2_test,y2_train,y2_test = train_test_split(X2.values, y2.values, random_state = 42)

Обучение модели для регрессии

In [13]:
sk_lin_reg = LinearRegression()
sk_lin_reg.fit(X2_train, y2_train)
sk_lin_reg_pred_res = sk_lin_reg.predict(X2_test)
sk_lin_reg_r2 = r2_score(y2_test, sk_lin_reg_pred_res)
sk_lin_reg_mae = mean_absolute_error(y2_test, sk_lin_reg_pred_res)
sk_lin_reg_mse = mean_squared_error(y2_test, sk_lin_reg_pred_res)

print(f'sk linear regressor r2: {sk_lin_reg_r2}')
print(f'sk linear regressor mae: {sk_lin_reg_mae}')
print(f'sk linear regressor mse: {sk_lin_reg_mse}')

sk linear regressor r2: 0.5381516327976896
sk linear regressor mae: 8059.199667645662
sk linear regressor mse: 130156343.5228364


## 3. Улучшение бейзлайна

### Сформулировать гипотезы (препроцессинг данных, визуализация данных, формирование новых признаков, подбор гиперпараметров на кросс-валидации и т.д.)

1. Формирование новых признаков: для задачи классификации можно создать новые признаки, комбинирующие в себе расходы и рейтинг участвующих в создании фильма людей соответственно.
2. Масштабирование данных во время предобработки
3. Подбор гиперпараметров
    - Для логистической регрессии
        - Инверсия регуляризации С
        - Метод оптимизации solver ('lbfgs', 'saga', 'liblinear' и т.д.)
    - Для линейной регрессии
        - Параметр регуляризации alpha

#### Задача классификации

1. Формирование новых признаков

In [14]:
data3 = df.copy()
data3["Expense"] = data3["Marketing expense"] + data3["Production expense"]
data3["Rating"] = data3["Lead_ Actor_Rating"] + data3["Lead_Actress_rating"] + data3["Director_rating"] + data3["Producer_rating"]

In [15]:
data4 = data3.copy()
data4 = data4.drop(["Marketing expense","Production expense","Lead_ Actor_Rating","Lead_Actress_rating","Director_rating","Producer_rating"],axis = 1)

In [16]:
X1_new = data4.drop('Start_Tech_Oscar', axis=1)
y1_new = data4['Start_Tech_Oscar']

X1_train_new,X1_test_new,y1_train_new,y1_test_new = train_test_split(X1_new.values, y1_new.values, random_state = 0)

Проверим, как повлияло введение новых признаков

In [17]:
sk_log_reg = LogisticRegression(max_iter=500, random_state=42)
sk_log_reg.fit(X1_train_new, y1_train_new)
sk_log_reg_pred_res = sk_log_reg.predict(X1_test_new)
sk_log_reg_accuracy = accuracy_score(y1_test_new, sk_log_reg_pred_res)
sk_log_reg_precision = precision_score(y1_test_new, sk_log_reg_pred_res)
sk_log_reg_recall = recall_score(y1_test_new, sk_log_reg_pred_res)
sk_log_reg_f1 = f1_score(y1_test_new, sk_log_reg_pred_res)

print(f'sk logistic regressor accuracy: {sk_log_reg_accuracy:}')
print(f'sk logistic regressor precision: {sk_log_reg_precision:}')
print(f'sk logistic regressor recall: {sk_log_reg_recall:}')
print(f'sk logistic regressor f1: {sk_log_reg_f1:}')

sk logistic regressor accuracy: 0.6535433070866141
sk logistic regressor precision: 0.7424242424242424
sk logistic regressor recall: 0.6447368421052632
sk logistic regressor f1: 0.6901408450704226


2. Масштабирование данных

In [18]:
scaler = StandardScaler()
X1_train_scaled = scaler.fit_transform(X1_train)
X1_test_scaled = scaler.transform(X1_test)

Проверим, как повлияло масштабирование данных

In [19]:
sk_log_reg = LogisticRegression(max_iter=500, random_state=42)
sk_log_reg.fit(X1_train_scaled, y1_train)
sk_log_reg_pred_res = sk_log_reg.predict(X1_test_scaled)
sk_log_reg_accuracy = accuracy_score(y1_test, sk_log_reg_pred_res)
sk_log_reg_precision = precision_score(y1_test, sk_log_reg_pred_res)
sk_log_reg_recall = recall_score(y1_test, sk_log_reg_pred_res)
sk_log_reg_f1 = f1_score(y1_test, sk_log_reg_pred_res)

print(f'sk logistic regressor accuracy: {sk_log_reg_accuracy:}')
print(f'sk logistic regressor precision: {sk_log_reg_precision:}')
print(f'sk logistic regressor recall: {sk_log_reg_recall:}')
print(f'sk logistic regressor f1: {sk_log_reg_f1:}')

sk logistic regressor accuracy: 0.6062992125984252
sk logistic regressor precision: 0.7096774193548387
sk logistic regressor recall: 0.5789473684210527
sk logistic regressor f1: 0.6376811594202899


3. Подбор гиперпараметров

In [25]:
from sklearn.model_selection import GridSearchCV

param_grid_class = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'max_iter': [n for n in range(200, 2100+1, 100)],
    'solver': ['lbfgs', 'liblinear'],
}

log_reg_class = LogisticRegression(random_state=42)
grid_search_class = GridSearchCV(log_reg_class, param_grid_class, cv=5, scoring='accuracy')
grid_search_class.fit(X1_train, y1_train)
best_knn_class = grid_search_class.best_estimator_

print("Лучшие параметры для классификации (HAR):", grid_search_class.best_params_)

y_pred_class = best_knn_class.predict(X1_test)
accuracy = accuracy_score(y1_test, y_pred_class)
precision = precision_score(y1_test, y_pred_class)
recall = recall_score(y1_test, y_pred_class)
f1 = f1_score(y1_test, y_pred_class)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

Лучшие параметры для классификации (HAR): {'C': 1000, 'max_iter': 200, 'solver': 'lbfgs'}
Accuracy: 0.6456692913385826
Precision: 0.7384615384615385
Recall: 0.631578947368421
F1 Score: 0.6808510638297872


#### Задача регрессии

2. Масштабирование данных

In [26]:
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

In [27]:
sk_lin_reg = LinearRegression()
sk_lin_reg.fit(X2_train_scaled, y2_train)
sk_lin_reg_pred_res = sk_lin_reg.predict(X2_test_scaled)
sk_lin_reg_r2 = r2_score(y2_test, sk_lin_reg_pred_res)
sk_lin_reg_mae = mean_absolute_error(y2_test, sk_lin_reg_pred_res)
sk_lin_reg_mse = mean_squared_error(y2_test, sk_lin_reg_pred_res)

print(f'sk linear regressor r2: {sk_lin_reg_r2}')
print(f'sk linear regressor mae: {sk_lin_reg_mae}')
print(f'sk linear regressor mse: {sk_lin_reg_mse}')

sk linear regressor r2: 0.5381516327976626
sk linear regressor mae: 8059.199667646258
sk linear regressor mse: 130156343.522844


3. Подбор гиперпараметров

In [28]:
from sklearn.linear_model import Ridge, Lasso

param_grid_reg = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'saga']
}

lin_reg = Ridge()
grid_search_reg = GridSearchCV(lin_reg, param_grid_reg, cv=5, scoring='r2')
grid_search_reg.fit(X2_train_scaled, y2_train)
best_lin_reg = grid_search_reg.best_estimator_

print("Лучшие параметры для Ridge регрессии:", grid_search_reg.best_params_)

y_pred_reg = best_lin_reg.predict(X2_test_scaled)
r2 = r2_score(y2_test, y_pred_reg)
mae = mean_absolute_error(y2_test, y_pred_reg)
mse = mean_squared_error(y2_test, y_pred_reg)

print(f"r2: {r2}")
print("mae:", mae)
print("mse:", mse)




param_grid_reg = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'max_iter': [n for n in range(200, 2100+1, 100)]
}

lin_reg = Lasso()
grid_search_reg = GridSearchCV(lin_reg, param_grid_reg, cv=5, scoring='r2')
grid_search_reg.fit(X2_train_scaled, y2_train)
best_lin_reg = grid_search_reg.best_estimator_

print("Лучшие параметры для Lasso регрессии:", grid_search_reg.best_params_)

y_pred_reg = best_lin_reg.predict(X2_test_scaled)
r2 = r2_score(y2_test, y_pred_reg)
mae = mean_absolute_error(y2_test, y_pred_reg)
mse = mean_squared_error(y2_test, y_pred_reg)

print(f"r2: {r2}")
print("mae:", mae)
print("mse:", mse)

Лучшие параметры для Ridge регрессии (CO2): {'alpha': 10, 'solver': 'lsqr'}
r2: 0.551214299209421
mae: 7974.9973669784895
mse: 126475072.74751987


C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.603e+10, tolerance: 1.074e+07
  model = cd_fast.enet_coordinate_descent(
C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.541e+10, tolerance: 1.060e+07
  model = cd_fast.enet_coordinate_descent(
C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or co

Лучшие параметры для Lasso регрессии (CO2): {'alpha': 100, 'max_iter': 500}
r2: 0.5554233957909451
mae: 7951.563175899328
mse: 125288881.21910925


### Выводы

Для задачи классификации ощутимого улучшения не произошло.

Для задачи регрессии подбор параметров позволил немного улучшить результат.

## 4. Имплементация алгоритма машинного обучения 

### Самостоятельная имплементация алгоритмов машинного обучения для классификации и регрессии

Реализация для классификации

In [29]:
class CustomLogisticRegression:
    def __init__(self, learn_rate=0.01, max_iter=200, penalty=None, C=1.0, tolerancy=10**(-4)):
        self.learn_rate = learn_rate
        self.max_iter = max_iter
        self.penalty = penalty
        self.C = C
        self.tolerancy = tolerancy

    def _calc_sigmoid(self, a):
        return 1 / (1 + np.exp(-a))

    def _run_iterations(self):
        for _ in range(self.max_iter):
            predictions = self._calc_sigmoid(np.dot(self.X_train, self.weights))
            errors = predictions - self.y_train
            gradient = np.dot(self.X_train.T, errors) / len(self.y_train)
            
            if self.penalty == 'l2':
                gradient += (1 / self.C) * self.weights
            elif self.penalty == 'l1':
                gradient += (1 / self.C) * np.sign(self.weights)
            
            self.weights -= self.learn_rate * gradient
            
            if np.linalg.norm(gradient) < self.tolerancy:
                break

    def fit(self, X_train, y_train):
        self.X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
        self.y_train = y_train
        self.weights = np.zeros(self.X_train.shape[1])
        
        self._run_iterations()

    def predict(self, X_test):
        X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))
        calculated_probabilities = self._calc_sigmoid(np.dot(X_test, self.weights))

        return (calculated_probabilities >= 0.5).astype(int)

Реализация для регресии

In [30]:
class CustomLinearRegression:
    def __init__(self, learn_rate=0.01, max_iter=200, penalty=None, alpha=0.0, tolerancy=10**(-4)):
        self.learn_rate = learn_rate
        self.max_iter = max_iter
        self.penalty = penalty
        self.alpha = alpha
        self.tolerancy = tolerancy
    
    def _run_iterations(self):
        for _ in range(self.max_iter):
            predictions = np.dot(self.X_train, self.weights)
            errors = predictions - self.y_train
            gradient = np.dot(self.X_train.T, errors) / len(self.y_train)
            
            if self.penalty == 'l2':
                gradient += (2 * self.alpha) * self.weights
            elif self.penalty == 'l1':
                gradient += self.alpha * np.sign(self.weights)
            
            self.weights -= self.learn_rate * gradient
            
            if np.linalg.norm(gradient) < self.tolerancy:
                break

    def fit(self, X_train, y_train):
        self.X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
        self.y_train = y_train
        self.weights = np.zeros(self.X_train.shape[1])

        self._run_iterations()

    def predict(self, X_test):
        X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))
        
        return np.dot(X_test, self.weights)

### Обучение имплементированных моделей

In [40]:
custom_log_reg = CustomLogisticRegression()
custom_log_reg.fit(X1_train, y1_train)
custom_log_reg_pred_res = custom_log_reg.predict(X1_test)
custom_log_reg_accuracy = accuracy_score(y1_test, custom_log_reg_pred_res)
custom_log_reg_precision = precision_score(y1_test, custom_log_reg_pred_res)
custom_log_reg_recall = recall_score(y1_test, custom_log_reg_pred_res)
custom_log_reg_f1 = f1_score(y1_test, custom_log_reg_pred_res)

print(f'custom logistic regressor accuracy: {custom_log_reg_accuracy:}')
print(f'custom logistic regressor precision: {custom_log_reg_precision:}')
print(f'custom logistic regressor recall: {custom_log_reg_recall:}')
print(f'custom logistic regressor f1: {custom_log_reg_f1:}')

custom logistic regressor accuracy: 0.47244094488188976
custom logistic regressor precision: 1.0
custom logistic regressor recall: 0.11842105263157894
custom logistic regressor f1: 0.2117647058823529


C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_11068\3826996408.py:10: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-a))


In [41]:
custom_lin_reg = CustomLinearRegression(learn_rate=0.01, max_iter=500)
custom_lin_reg.fit(X2_train, y2_train)
custom_lin_reg_pred_res = custom_lin_reg.predict(X2_test)
custom_lin_reg_r2 = r2_score(y2_test, custom_lin_reg_pred_res)
custom_lin_reg_mae = mean_absolute_error(y2_test, custom_lin_reg_pred_res)
custom_lin_reg_mse = mean_squared_error(y2_test, custom_lin_reg_pred_res)

print(f'custom linear regressor r2: {custom_lin_reg_r2}')
print(f'custom linear regressor mae: {custom_lin_reg_mae}')
print(f'custom linear regressor mse: {custom_lin_reg_mse}')

C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_11068\958302101.py:20: RuntimeWarning: invalid value encountered in subtract
  self.weights -= self.learn_rate * gradient


ValueError: Input contains NaN.

### Выводы

С сипользованием собственных моделей возникают проблемы. Логистическая регрессия показывает неудовлетворительные результаты, линейная регрессия выдает ошибку, т.к. в процессе вычислений появляются большие увеличивающиеся числа и в какой-то момент становятся нечисловыми значениями "бесконечность". Возможно, ситуацию исправит масштабирование данных.

### Обучение имплементированных моделей в улучшенном бейзлайне

#### Задача классификации

Обязательно применяем масштабирование

In [43]:
custom_log_reg = CustomLogisticRegression()
custom_log_reg.fit(X1_train_scaled, y1_train)
custom_log_reg_pred_res = custom_log_reg.predict(X1_test_scaled)
custom_log_reg_accuracy = accuracy_score(y1_test, custom_log_reg_pred_res)
custom_log_reg_precision = precision_score(y1_test, custom_log_reg_pred_res)
custom_log_reg_recall = recall_score(y1_test, custom_log_reg_pred_res)
custom_log_reg_f1 = f1_score(y1_test, custom_log_reg_pred_res)

print(f'custom logistic regressor accuracy: {custom_log_reg_accuracy:}')
print(f'custom logistic regressor precision: {custom_log_reg_precision:}')
print(f'custom logistic regressor recall: {custom_log_reg_recall:}')
print(f'custom logistic regressor f1: {custom_log_reg_f1:}')

custom logistic regressor accuracy: 0.5669291338582677
custom logistic regressor precision: 0.6363636363636364
custom logistic regressor recall: 0.6447368421052632
custom logistic regressor f1: 0.6405228758169935


Попробуем добавить к масштабированию формирование новых признаков

In [45]:
X1_train_new_scaled = scaler.fit_transform(X1_train_new)
X1_test_new_scaled = scaler.transform(X1_test_new)

custom_log_reg = CustomLogisticRegression()
custom_log_reg.fit(X1_train_new_scaled, y1_train_new)
custom_log_reg_pred_res = custom_log_reg.predict(X1_test_new_scaled)
custom_log_reg_accuracy = accuracy_score(y1_test_new, custom_log_reg_pred_res)
custom_log_reg_precision = precision_score(y1_test_new, custom_log_reg_pred_res)
custom_log_reg_recall = recall_score(y1_test_new, custom_log_reg_pred_res)
custom_log_reg_f1 = f1_score(y1_test_new, custom_log_reg_pred_res)

print(f'custom logistic regressor accuracy: {custom_log_reg_accuracy:}')
print(f'custom logistic regressor precision: {custom_log_reg_precision:}')
print(f'custom logistic regressor recall: {custom_log_reg_recall:}')
print(f'custom logistic regressor f1: {custom_log_reg_f1:}')

custom logistic regressor accuracy: 0.5511811023622047
custom logistic regressor precision: 0.620253164556962
custom logistic regressor recall: 0.6447368421052632
custom logistic regressor f1: 0.6322580645161291


Попробуем добавить к масштабированию подбор параметров

In [48]:
from itertools import product

best_params_classification = {}
best_metrics_classification = {"accuracy": 0}

learn_rate_variants = [0.001, 0.01, 0.1, 0.5]
max_iter_variants = [n for n in range(200, 2100+1, 200)]
penalty_variants = [None, 'l1', 'l2']
C_variants = [0.1, 0.5, 1.0, 5.0]

for learn_rate, max_iter, penalty, C in product(learn_rate_variants, max_iter_variants, penalty_variants, C_variants):
    custom_log_reg = CustomLogisticRegression(learn_rate=learn_rate, max_iter=max_iter, penalty=penalty, C=C)
    custom_log_reg.fit(X1_train_scaled, y1_train)
    pred_res = custom_log_reg.predict(X1_test_scaled)

    accuracy = accuracy_score(y1_test, pred_res)
    precision = precision_score(y1_test, pred_res)
    recall = recall_score(y1_test, pred_res)
    f1 = f1_score(y1_test, pred_res)

    if accuracy > best_metrics_classification["accuracy"]:
        best_params_classification = {
            "learn_rate": learn_rate,
            "max_iter": max_iter,
            "penalty": penalty,
            "C": C
        }
        best_metrics_classification = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

print("Лучшие параметры для классификации:", best_params_classification)
print("Метрики для классификации:", best_metrics_classification)

C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_11068\3826996408.py:10: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-a))
C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_11068\3826996408.py:10: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-a))
C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_11068\3826996408.py:10: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-a))
C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_11068\3826996408.py:19: RuntimeWarning: overflow encountered in multiply
  gradient += (1 / self.C) * self.weights
C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_11068\3826996408.py:10: RuntimeWarning: 

Лучшие параметры для классификации: {'learn_rate': 0.5, 'max_iter': 400, 'penalty': 'l1', 'C': 0.5}
Метрики для классификации: {'accuracy': 0.6456692913385826, 'precision': 0.7719298245614035, 'recall': 0.5789473684210527, 'f1': 0.6616541353383459}


C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_11068\3826996408.py:10: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-a))
C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_11068\3826996408.py:19: RuntimeWarning: overflow encountered in multiply
  gradient += (1 / self.C) * self.weights
C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


#### Задача регрессии

Обязательно применяем масштабирование

In [39]:
custom_lin_reg = CustomLinearRegression(learn_rate=0.01, max_iter=500)
custom_lin_reg.fit(X2_train_scaled, y2_train)
custom_lin_reg_pred_res = custom_lin_reg.predict(X2_test_scaled)
custom_lin_reg_r2 = r2_score(y2_test, custom_lin_reg_pred_res)
custom_lin_reg_mae = mean_absolute_error(y2_test, custom_lin_reg_pred_res)
custom_lin_reg_mse = mean_squared_error(y2_test, custom_lin_reg_pred_res)

print(f'custom linear regressor r2: {custom_lin_reg_r2}')
print(f'custom linear regressor mae: {custom_lin_reg_mae}')
print(f'custom linear regressor mse: {custom_lin_reg_mse}')

custom linear regressor r2: 0.5479525527926385
custom linear regressor mae: 7930.44314299149
custom linear regressor mse: 127394285.66945522


Добавим к масштабированию подбор параметров

In [38]:
from itertools import product

best_params_classification = {}
best_metrics_classification = {"r2": 0}

learn_rate_variants = [0.001, 0.01, 0.1, 0.5]
max_iter_variants = [n for n in range(200, 700+1, 100)]
penalty_variants = [None, 'l1', 'l2']

for learn_rate, max_iter, penalty in product(learn_rate_variants, max_iter_variants, penalty_variants):
    custom_lin_reg = CustomLinearRegression(learn_rate=learn_rate, max_iter=max_iter, penalty=penalty)
    custom_lin_reg.fit(X2_train_scaled, y2_train)
    pred_res = custom_lin_reg.predict(X2_test_scaled)

    r2 = r2_score(y2_test, pred_res)
    mae = mean_absolute_error(y2_test, pred_res)
    mse = mean_squared_error(y2_test, pred_res)

    if r2 > best_metrics_classification["r2"]:
        best_params_classification = {
            "learn_rate": learn_rate,
            "max_iter": max_iter,
            "penalty": penalty
        }
        best_metrics_classification = {
            "r2": r2,
            "mae": mae,
            "mse": mse
        }

print("Лучшие параметры для классификации:", best_params_classification)
print("Метрики для классификации:", best_metrics_classification)

Лучшие параметры для классификации: {'learn_rate': 0.01, 'max_iter': 300, 'penalty': None}
Метрики для классификации: {'r2': 0.5600548968039499, 'mae': 7478.550265751896, 'mse': 123983649.28654522}


C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_regression.py:927: RuntimeWarning: overflow encountered in square
  numerator = (weight * (y_true - y_pred) ** 2).sum(axis=0, dtype=np.float64)
C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_regression.py:446: RuntimeWarning: overflow encountered in square
  output_errors = np.average((y_true - y_pred) ** 2, axis=0, weights=sample_weight)
C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_regression.py:927: RuntimeWarning: overflow encountered in square
  numerator = (weight * (y_true - y_pred) ** 2).sum(axis=0, dtype=np.float64)
C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_regression.py:446: RuntimeWarning: overflow encountered in square
  output_errors = np.average((y_true - y_pred) ** 2, axis=0, weights=sample_weight)
C:\Users\Gigabyte\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_re

### Выводы

Как и для встроенных моделей, удалось улучшить результаты работы.
В целом, результаты встроеных и собственных моделей в улучшенных бейзлайнах схожи.